# Word2Vec with TensorFlow / Keras

Beginner-friendly version of the PyTorch reference notebook.

Why Keras?
- No need to write the training loop manually.
- No `loss.backward()`, `optimizer.zero_grad()`, `optimizer.step()`.
- Just `model.compile(...)` and `model.fit(...)`.

We will cover:
1. Co-occurrence matrix (no neural network — pure counting)
2. **CBOW** (Continuous Bag of Words) in Keras
3. **Skip-Gram** in Keras

## 1. Co-Occurrence Matrix

Before learning embeddings, let's see how words *co-occur* in a small corpus.

**Intuition:** if two words frequently appear near each other (within a small
window), they are probably related in meaning. A co-occurrence matrix simply
counts how often word A appears in the neighborhood of word B.

In [ ]:
# A tiny toy corpus — just two sentences so the matrix is easy to read.
sentences = [
    "the cat sat on the mat",
    "dogs bark at night",
    #"the sun rises in the east",
    #"birds fly high in the sky",
    #"children play games after school"
]

# Step 1: tokenize
# 'tokenize' = split a sentence into individual words (tokens).
# We also lowercase everything so 'The' and 'the' are treated as the same word.
tokenized = [s.lower().split() for s in sentences]

# Step 2: build the vocabulary (the unique set of words across all sentences)
# We sort it so the order is deterministic — same vocab order every time we run.
vocab = sorted(set(word for sent in tokenized for word in sent))

# Step 3: assign each word a unique integer id (and keep the reverse mapping too)
# Neural nets can't process strings — they need numbers. word_to_id does that mapping.
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
vocab_size = len(vocab)

print("Vocabulary:", word_to_id)

In [ ]:
from collections import defaultdict
import numpy as np

# 'window_size' = how many words to the LEFT and RIGHT of a center word we look at.
# E.g., window_size=2 means we look at 2 words before + 2 words after the center word.
window_size = 2

# defaultdict(float) lets us write `cooc[(a, b)] += 1.0` without first checking
# whether the key exists. Missing keys default to 0.0.
cooc = defaultdict(float)

# Walk through every sentence...
for sentence in tokenized:
    # ...for every word in the sentence (we call this the 'center' word)...
    for center_idx, center_word in enumerate(sentence):
        center_id = word_to_id[center_word]

        # ...look at offsets from -window_size to +window_size (excluding 0).
        # offset = -2 means the word 2 positions to the LEFT of the center.
        # offset = +1 means the word 1 position to the RIGHT.
        for offset in range(-window_size, window_size + 1):
            if offset == 0:
                # offset 0 IS the center word itself — skip, we don't pair a word with itself.
                continue

            context_idx = center_idx + offset

            # Make sure we don't fall off the start/end of the sentence.
            if 0 <= context_idx < len(sentence):
                context_word = sentence[context_idx]
                context_id = word_to_id[context_word]

                # Increment the count: 'center_word appeared near context_word once more'.
                cooc[(center_id, context_id)] += 1.0

In [ ]:
import pandas as pd

# Make pandas show ALL rows/columns (otherwise it truncates large tables with '...').
pd.options.display.max_columns = None
pd.options.display.max_rows = None

# Start with a square zero matrix of shape (vocab_size, vocab_size).
# Row i, column j  = 'how often word j appeared near word i'.
matrix = np.zeros((vocab_size, vocab_size))

# Fill in the counts we collected above.
for (i, j), v in cooc.items():
    matrix[i, j] = round(v, 2)

# Wrap it in a DataFrame so the rows/columns are labeled with the actual words.
df = pd.DataFrame(matrix, columns=vocab, index=vocab)
df.fillna(0)

## 2. CBOW (Continuous Bag of Words) — Keras version

**Idea:** Given the *context words* around a target, predict the *target word*.

Example with window=2:

`['the', 'cat', 'on', 'the'] -> 'sat'`

**How the Keras model works:**
1. `Embedding` layer turns each context word id into a dense vector (the "meaning" of the word).
2. `GlobalAveragePooling1D` averages the context vectors into a single vector.
3. `Dense(vocab_size, softmax)` predicts which word in the vocab is the target.

Keras handles the loss, gradients, and weight updates for us via `model.fit` —
no need to write `loss.backward()` or `optimizer.step()` ourselves.

In [ ]:
# numpy: numerical arrays. Keras/TensorFlow expects inputs as numpy arrays (or tensors).
import numpy as np
# tensorflow: the deep-learning framework. Keras lives inside tf as `tf.keras`.
import tensorflow as tf
# 'layers' is a shortcut to all the layer classes (Dense, Embedding, etc.).
from tensorflow.keras import layers
# `chain` flattens a list of lists into a single iterable. Handy for vocab building.
from itertools import chain

print("TensorFlow version:", tf.__version__)

In [ ]:
# A slightly bigger toy corpus so the model has a few patterns to learn from.
# Real Word2Vec is trained on BILLIONS of words; this is just for demonstration.
sentences = [
    "the cat sat on the mat in the evening quietly",
    "dogs bark loudly when they see strangers near the gate",
    "birds fly in the sky and sing songs in the morning",
    "the sun rises in the east and sets in the west",
    "children play games in the park after school hours",
    "the teacher teaches maths and science in the classroom",
    "books are kept neatly on the shelf by the librarian",
    "the gardener waters the plants in the garden daily",
    "parents drop their kids at school every morning",
    "students read and write silently during the study hour",
]

# Tokenize: split each sentence into lowercase words.
tokenized = [s.lower().split() for s in sentences]

# Build the vocabulary by flattening all sentences into one big list of words,
# then taking the unique set, then sorting (for reproducibility).
vocab = sorted(set(chain(*tokenized)))

# Word <-> integer-id mappings (neural networks need numeric inputs).
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for w, i in word_to_ix.items()}
vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)

In [ ]:
# For CBOW, every training example is: (context words, target word).
# We slide a 'window' across each sentence and at every position we record:
#   - the 2 words BEFORE + 2 words AFTER the center word -> context
#   - the center word itself -> target

window_size = 2  # 2 words on each side -> 4 context words per example

def generate_cbow_pairs(tokenized_sentences, window_size=2):
    pairs = []
    for sentence in tokenized_sentences:
        # We start at index `window_size` (so we have enough words on the LEFT)
        # and stop `window_size` words before the end (enough words on the RIGHT).
        for i in range(window_size, len(sentence) - window_size):
            # Build the context: window_size words to the left + window_size words to the right.
            # The reversed range on the left keeps left-context in original order.
            context = [sentence[i - j] for j in range(window_size, 0, -1)] + \
                      [sentence[i + j] for j in range(1, window_size + 1)]
            target = sentence[i]
            pairs.append((context, target))
    return pairs

cbow_data = generate_cbow_pairs(tokenized, window_size)

# Print the first few pairs to sanity-check what we built.
print("Sample CBOW pairs (context -> target):")
for context, target in cbow_data[:5]:
    print(context, "->", target)

In [ ]:
# The model can't take strings as input, so we convert words -> integer ids.
# X[i] is a list of 4 ids (the context words for example i).
# y[i] is a single id (the target word for example i).

# dtype=np.int32 is required because Embedding layers expect integer indices.
X = np.array([[word_to_ix[w] for w in ctx] for ctx, _ in cbow_data], dtype=np.int32)
y = np.array([word_to_ix[tgt] for _, tgt in cbow_data], dtype=np.int32)

print("X shape:", X.shape, "  (num_examples, 2*window_size)")
print("y shape:", y.shape)
print("First example -> X:", X[0], " y:", y[0])

In [ ]:
# 'embedding_dim' = size of each word's vector representation.
# Bigger = more expressive, but more parameters to learn. 50 is fine for a toy corpus.
embedding_dim = 50

# Sequential = a simple stack of layers, one after another.
cbow_model = tf.keras.Sequential([
    # Input: each example is a list of 2*window_size = 4 word ids.
    layers.Input(shape=(2 * window_size,)),

    # Embedding layer: a lookup table of shape (vocab_size, embedding_dim).
    # It converts each word id into a dense vector of size embedding_dim.
    # Output shape: (batch_size, 4, embedding_dim).
    # We give it a name so we can grab the learned weights later.
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, name="embedding"),

    # GlobalAveragePooling1D: averages the 4 context vectors into ONE vector.
    # Output shape: (batch_size, embedding_dim).
    # This is the 'bag of words' part — the order of context words doesn't matter,
    # we just average them all together.
    layers.GlobalAveragePooling1D(),

    # Final dense layer: project the averaged context vector to a probability
    # over the entire vocabulary. softmax makes the outputs sum to 1.
    # Output shape: (batch_size, vocab_size).
    layers.Dense(vocab_size, activation="softmax"),
])

# compile = tell Keras HOW to train: which optimizer, which loss, which metrics.
#   - Adam: a good default optimizer (does the gradient updates for us)
#   - sparse_categorical_crossentropy: standard classification loss when the
#     target is an integer id (NOT a one-hot vector). Saves us from having to
#     one-hot encode y.
#   - accuracy: helpful sanity-check metric (prediction == target?).
cbow_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# summary() prints the layer-by-layer architecture and parameter count.
cbow_model.summary()

In [ ]:
# THIS IS THE WHOLE TRAINING LOOP. ONE LINE.
# Compare to PyTorch where you wrote: zero_grad, forward, loss, backward, step.
# Keras does all of that internally.
#
# Args:
#   X, y       -> training inputs and targets
#   epochs=100 -> go through the full dataset 100 times
#   batch_size=8 -> update weights after every 8 examples
#   shuffle=True -> shuffle the data each epoch (like random.shuffle in PyTorch)
#   verbose=0   -> silent; we'll print loss ourselves below
history = cbow_model.fit(
    X, y,
    epochs=100,
    batch_size=8,
    shuffle=True,
    verbose=0,
)

# `history.history` is a dict: { 'loss': [...], 'accuracy': [...] } — one value per epoch.
# We print every 10th epoch to mimic the original PyTorch notebook's output style.
for epoch in range(0, 100, 10):
    print(f"Epoch {epoch} Loss: {history.history['loss'][epoch]:.4f}")

In [ ]:
# After training, the Embedding layer's weight matrix IS the word2vec embeddings.
# get_weights() returns a list; for an Embedding layer the only weight is the
# lookup table of shape (vocab_size, embedding_dim).
embedding_matrix = cbow_model.get_layer("embedding").get_weights()[0]

# To get the vector for a specific word, look up its row in the matrix.
word = "school"
vec = embedding_matrix[word_to_ix[word]]
print(f"Embedding vector for '{word}' (dim={vec.shape[0]}):\n", vec)

In [ ]:
# Use the trained model to make a prediction.
# Given a context, what does the model think the target word is?

context_words = ["the", "cat", "on", "the"]

# model.predict expects a BATCH of examples, even if there's just one.
# So we wrap our 4-id list inside another list -> shape (1, 4).
context_ids = np.array([[word_to_ix[w] for w in context_words]])

# predict() returns probabilities over the whole vocabulary, shape (1, vocab_size).
# [0] picks the first (and only) example in the batch.
probs = cbow_model.predict(context_ids, verbose=0)[0]

# argmax picks the index with the highest probability — the model's top guess.
predicted_id = probs.argmax()
print(f"Context {context_words} -> predicted target: '{ix_to_word[predicted_id]}'")

## 3. Skip-Gram — Keras version

**Idea:** Given the *center word*, predict each *context word* around it.
It's the mirror image of CBOW.

Example with window=2 for sentence `"the cat sat on the mat"`:

```
center 'cat' -> context 'the'
center 'cat' -> context 'sat'
center 'cat' -> context 'on'
```

**How the Keras model works:**
1. `Embedding` layer turns the center word id into a vector.
2. `Dense(vocab_size, softmax)` predicts which word is in its context.

Notice each (center, context) pair is one training example — Skip-Gram
produces a LOT more examples than CBOW from the same corpus.

In [ ]:
# Build (center, context) pairs.
# For each word in each sentence, pair it with EVERY word inside its window.

def generate_skipgram_pairs(tokenized_sentences, window_size=2):
    pairs = []
    for tokens in tokenized_sentences:
        sent_len = len(tokens)

        # Treat every word in the sentence as a potential center word.
        for idx, center in enumerate(tokens):
            # Compute the window boundaries, clamped to the sentence edges.
            # max(.., 0) prevents going off the LEFT side (negative index).
            # min(.., sent_len) prevents going off the RIGHT side.
            start = max(idx - window_size, 0)
            end = min(idx + window_size + 1, sent_len)

            # LEFT context: words from `start` up to (but not including) the center.
            pairs.extend((center, ctx) for ctx in tokens[start:idx])
            # RIGHT context: words from just after the center up to `end`.
            pairs.extend((center, ctx) for ctx in tokens[idx + 1:end])
    return pairs

skipgram_data = generate_skipgram_pairs(tokenized, window_size)

print("Sample Skip-Gram pairs (center -> context):")
for center, ctx in skipgram_data[:5]:
    print(center, "->", ctx)

In [ ]:
# Convert pairs (center_word, context_word) -> (center_id, context_id).
# X_sg = inputs (just the center word — a SINGLE id per example).
# y_sg = targets (the context word id we want the model to predict).

X_sg = np.array([word_to_ix[c] for c, _ in skipgram_data], dtype=np.int32)
y_sg = np.array([word_to_ix[ctx] for _, ctx in skipgram_data], dtype=np.int32)

print("X_sg shape:", X_sg.shape, "  (just the center word id)")
print("y_sg shape:", y_sg.shape)

In [ ]:
embedding_dim = 50

skipgram_model = tf.keras.Sequential([
    # Input: a single integer per example (the center word's id).
    # shape=(1,) means each example is a 1-element vector.
    layers.Input(shape=(1,)),

    # Embedding turns the id into a vector of size embedding_dim.
    # Because input length is 1, output shape is (batch, 1, embedding_dim).
    layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, name="embedding"),

    # Flatten removes the extra '1' dimension: (batch, 1, embed_dim) -> (batch, embed_dim).
    # We need this because Dense expects a flat vector per example.
    layers.Flatten(),

    # Predict probabilities over the whole vocabulary —
    # "how likely is each word to be in this center word's context?"
    layers.Dense(vocab_size, activation="softmax"),
])

# Same compile setup as CBOW — only the data and architecture differ.
skipgram_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

skipgram_model.summary()

In [ ]:
# Train the Skip-Gram model. Same one-line training loop as CBOW.
# Note: Skip-Gram usually has MORE training pairs than CBOW from the same corpus,
# so the loss numbers per epoch will look much bigger (it's a sum over more examples).
history_sg = skipgram_model.fit(
    X_sg, y_sg,
    epochs=100,
    batch_size=8,
    shuffle=True,
    verbose=0,
)

for epoch in range(0, 100, 10):
    print(f"Epoch {epoch} Loss: {history_sg.history['loss'][epoch]:.4f}")

In [ ]:
# Just like CBOW: pull out the Embedding layer's weights — those ARE the
# learned word vectors. Same shape as before: (vocab_size, embedding_dim).
embedding_matrix_sg = skipgram_model.get_layer("embedding").get_weights()[0]

target_word = "school"
vec = embedding_matrix_sg[word_to_ix[target_word]]
print(f"Embedding for '{target_word}':\n{vec}")

In [ ]:
# Bonus: find words most similar to a given word using cosine similarity.
#
# Cosine similarity measures the ANGLE between two vectors:
#   1.0  = same direction (very similar)
#   0.0  = perpendicular (unrelated)
#  -1.0  = opposite direction
# Formula: cos(a, b) = (a . b) / (||a|| * ||b||)

def most_similar(word, top_k=5, matrix=embedding_matrix_sg):
    # 1. Look up the vector for our query word.
    vec = matrix[word_to_ix[word]]

    # 2. Compute the denominator: ||a|| * ||b|| for every word in the vocab.
    #    + 1e-9 is a tiny number to avoid divide-by-zero.
    norms = np.linalg.norm(matrix, axis=1) * np.linalg.norm(vec) + 1e-9

    # 3. matrix @ vec computes the DOT product of `vec` with every row of `matrix`.
    #    Dividing by norms gives the cosine similarity for each word.
    sims = matrix @ vec / norms

    # 4. Sort by similarity DESCENDING (argsort gives ascending, so we reverse with [::-1]).
    best = sims.argsort()[::-1]

    # 5. Skip the query word itself (it would always be most similar to itself!),
    #    then return the top_k as (word, similarity) pairs.
    return [(ix_to_word[i], float(sims[i])) for i in best if i != word_to_ix[word]][:top_k]

# With a tiny corpus, results will look noisy — but the mechanism is correct.
print("Most similar to 'school':", most_similar("school"))

## Summary: PyTorch vs Keras

| Step | PyTorch (original) | Keras (this notebook) |
|---|---|---|
| Define model | subclass `nn.Module`, write `forward` | `tf.keras.Sequential([...])` |
| Loss + optimizer | `nn.CrossEntropyLoss()`, `optim.Adam(...)` | `model.compile(loss=..., optimizer=...)` |
| Training loop | manual: `zero_grad`, `backward`, `step` | one line: `model.fit(X, y, epochs=...)` |
| Get embeddings | `model.embeddings.weight[id]` | `model.get_layer('embedding').get_weights()[0][id]` |

Same model, same idea — just a friendlier API for beginners.